# Import Dependencies

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Getting dataset

In [ ]:
train_data = pd.read_csv('/kaggle/input/digit-recognizer/train.csv')
test_data = pd.read_csv('/kaggle/input/digit-recognizer/test.csv')

X = train_data.drop('label', axis=1).to_numpy() / 225.0
y = train_data['label'].to_numpy()
X_test = test_data.to_numpy() / 225.0

## Plot some samples

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=5,
                      sharex=True, sharey=True)
ax = ax.flatten()
classes = len(sorted(train_data.label.unique()))
for idx in range(classes):
    img = X[y == idx][0].reshape(28,28)
    ax[idx].imshow(img, cmap='Greys')
    ax[idx].set_title(idx)
ax[0].set_xticks([])
ax[0].set_yticks([])
plt.tight_layout()
plt.show()    

## Split data for training and val subsets

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, shuffle=True, random_state=123)
print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"X_val: {X_val.shape} | y_val: {y_val.shape} \nX_test: {X_test.shape}")

# Creating Simple NN

## Utility functions

In [ ]:
def relu(x):
    return np.maximum(0.0, x)

def relu_derivative(x):
    return (x>0).astype(float)

def softmax(logits):
    exps = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    return exps / np.sum(exps, axis=1, keepdims=True)

def softCrossEntropy(logits, y_true):
    probs = softmax(logits)
    correct_probs = probs[np.arange(len(y_true)), y_true]
    loss = -np.mean(np.log(correct_probs))
    return loss, probs

def accuracy(targets, predicated_labels):
    return np.mean(predicted_labels == targets)

In [ ]:
def minibatch_generator(X, y=None, mb_size=32, shuffle=True):
    indices = np.arange(X.shape[0])
    if shuffle:
        np.random.shuffle(indices)
        
    for start_idx in range(0, indices.shape[0], mb_size):
        batch_idx  = indices[start_idx:start_idx + mb_size]
        if y is not None:
            yield X[batch_idx], y[batch_idx]
        else:
            yield X[batch_idx]

In [ ]:
def compute_loss_and_acc(nnet, X, y, num_lables=10, minibatch_size=32):
    loss_, correct_pred, num_examples = 0., 0, 0
    mb_gen = minibatch_generator(X, y, minibatch_size)
    for i, (features, targets) in enumerate(mb_gen):
        logits = nnet.forward(features)
        loss, probs = softCrossEntropy(logits, targets)
        predicted_labels = np.argmax(probs, axis=1)
        correct_pred += (predicted_labels == targets).sum()
        num_examples += targets.shape[0]
        loss_ += loss
        
    loss_ = loss_/i
    acc = correct_pred/num_examples
    return loss_, acc*100

## Constructing NN class

In [ ]:

class MNISTNet:
    def __init__(self, num_features, num_hidden, num_classes, seed=123):
        self.num_classes = num_classes
        rng = np.random.RandomState(seed)

        # Hidden layer
        self.weight_ = rng.normal(0.0, 0.1, size=(num_hidden, num_features))
        self.bias_ = np.zeros(num_hidden)

        # Output layer
        self.weight_out = rng.normal(0.0, 0.1, size=(num_classes, num_hidden))
        self.bias_out = np.zeros(num_classes)

        # Initialize gradient buffers
        self.zero_grad()

    def zero_grad(self):
        self.dW_out = np.zeros_like(self.weight_out)
        self.db_out = np.zeros_like(self.bias_out)
        self.dW_hidden = np.zeros_like(self.weight_)
        self.db_hidden = np.zeros_like(self.bias_)

    def forward(self, x):
        # Hidden layer
        self.z_hidden = np.dot(x, self.weight_.T) + self.bias_
        self.a_hidden = relu(self.z_hidden)

        # Output layer
        self.z_out = np.dot(self.a_hidden, self.weight_out.T) + self.bias_out
        logits = self.z_out
        return logits

    def backward(self, x, logits, y_true):
        n_samples = x.shape[0]

        # softmax + cross-entropy grad
        probs = softmax(logits)
        probs[np.arange(n_samples), y_true] -= 1
        delta_out = probs / n_samples

        # Gradient for output layer
        self.dW_out = np.dot(delta_out.T, self.a_hidden)
        self.db_out = np.sum(delta_out, axis=0)

        # Backprop into hidden layer
        delta_hidden = np.dot(delta_out, self.weight_out) * relu_derivative(self.z_hidden)

        # Gradients for hidden layer
        self.dW_hidden = np.dot(delta_hidden.T, x)
        self.db_hidden = np.sum(delta_hidden, axis=0)
        
    def step(self, learning_rate=0.01):
        # Update Weights and Biases (simple SGD)
        self.weight_out -= learning_rate * self.dW_out
        self.bias_out -= learning_rate * self.db_out
        self.weight_ -= learning_rate * self.dW_hidden
        self.bias_ -= learning_rate * self.db_hidden

## Intantiating MNISTNet

In [ ]:
model = MNISTNet(num_features = 784,
                num_hidden=50,
                num_classes=10)

# Model training and validation

In [ ]:
%%time
np.random.seed(123)
num_epochs = 100
mb_size = 32

def train(model, X_train, y_train, X_val, y_val, num_epochs, learning_rate=0.01):
    epoch_train_loss, epoch_val_loss, epoch_train_acc, epoch_val_acc = [], [], [], []
    for epoch in range(num_epochs):
        mb_gen = minibatch_generator(X_train, y_train, mb_size)
        for X, y in mb_gen:
            logits = model.forward(X)
            loss, probs = softCrossEntropy(logits, y)
            model.zero_grad()
            model.backward(X, logits, y)
            model.step(learning_rate)
        
        train_loss, train_acc = compute_loss_and_acc(model, X_train, y_train)
        val_loss, val_acc = compute_loss_and_acc(model, X_val, y_val)
        
        epoch_train_acc.append(train_acc)
        epoch_val_acc.append(val_acc)
        epoch_train_loss.append(train_loss)
        epoch_val_loss.append(val_loss)
        if epoch % 10 == 0:
            print(f"Epoch: {epoch+10} | Loss: {train_loss:.2f} | Acc: {train_acc:.2f} | Val loss: {val_loss:.2f} | Val acc: {val_acc:.2f}")
    return epoch_train_loss, epoch_train_acc, epoch_val_loss, epoch_val_acc    


epoch_train_loss, epoch_train_acc, epoch_val_loss, epoch_val_acc = train(model, X_train, y_train, X_val, y_val, num_epochs=num_epochs, learning_rate=0.01)

## Plot Model's Loss and Accuracy

In [ ]:
epochs = range(len(epoch_train_loss))
plt.figure(figsize=(15,7))

plt.subplot(1,2,1)
plt.plot(epochs, epoch_train_loss, label='train loss')
plt.plot(epochs, epoch_val_loss, label='val loss')
plt.title('Loss')
plt.xlabel('Epochs')
plt.legend()

plt.subplot(1,2,2)
plt.plot(epochs, epoch_train_acc, label='train accuracy')
plt.plot(epochs, epoch_val_acc, label='val accuracy')
plt.title('Accuracy')
plt.xlabel('Epochs')
plt.legend();

# Predictions

In [ ]:
predictions = []
for x in minibatch_generator(X_test, mb_size=mb_size, shuffle=False):
    logits = model.forward(x)
    probs = softmax(logits)
    predictions.append(np.argmax(probs, axis=1))

predictions = np.concatenate(predictions)

submission = pd.DataFrame({"ImageId" : range(1, len(predictions) +1),
                          "Label": predictions})
submission.to_csv("submission.csv", index=False)
print(f"[INFO] Submission created successfully")